# General Data Science Track — Session 07
## Predict the next telemetry visit without patient leakage
**MED-EL research preparation · Estimated core time: 60 minutes**

## Why this matters
Research question: **“For a patient absent from model development, can current telemetry predict the mean diagonal proxy at the next scheduled visit?”** Refresh scikit-learn around a realistic target and a strong persistence baseline. Multiple visit pairs belong to the same patient, so both holdout allocation and cross-validation must respect patient identity.

## Learning objectives
- Define prediction time, a future outcome, and features available at prediction time.
- Keep every patient's visit pairs in one partition in both holdout and CV.
- Fit imputation, categorical encoding, scaling, and ridge regression inside a pipeline.
- Compare against persistence with MAE and visual error analysis.
- Separate model selection from final evaluation and state the limits of a synthetic benchmark.

## R-to-Python bridge
| Familiar concept | scikit-learn |
|---|---|
| grouped data split | `GroupShuffleSplit` |
| grouped resampling | `GroupKFold` |
| preprocessing recipe | `ColumnTransformer` + `Pipeline` |
| penalized linear regression | `Ridge(alpha=...)` |
| resampled tuning | `GridSearchCV(...).fit(X, y, groups=...)` |
| absolute prediction error | `mean_absolute_error(y, prediction)` |

Use integer positional indices from splitters with `.iloc[...]`. A scikit-learn pipeline learns transformations during `.fit()` and reuses them during `.predict()`. Lower MAE is better; scikit-learn's `neg_mean_absolute_error` score has the opposite sign.

## Suggested pacing
Tasks 1–3: 17 min · Tasks 4–6: 22 min · Tasks 7–8: 13 min · Retrieval/reasoning/reflection: 8 min.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)** (`.venv`). All files already exist in `data/ci_telemetry/`; no downloads or prior notebook variables are needed. Read [the data dictionary](data/ci_telemetry/README.md) when a field is unfamiliar.

**All data are synthetic.** There are 96 patients with exactly one implant each, 360 observed visits, and 144 directed cell records per visit. Matrix rows stimulate; columns record. Contacts 1–12 have an index order but no claimed anatomical direction. These are teaching conventions, not a description of MED-EL's internal export. `voltage_mv / stim_current_ua` gives a synthetic V/I proxy in kΩ; it is not automatically the manufacturer's clinical impedance measurement. Preserve the original tables. Write short explanations in new markdown cells or print them. Exercise cells are intentionally empty.

Load `prediction_pairs.csv` (264×22). Each row pairs a current visit with its next observed scheduled visit (0→1, 1→3, or 3→6). The target is `target_diag_mean_kohm`, measured later. There are 96 patients; each has 2–3 pairs. Target visit IDs/windows are for auditing only. Exact future timing and protocol are not available as predictors. `planned_horizon_months` is known from the schedule.

This benchmark applies to observed next visits. It does not model loss to follow-up, clinical outcomes, or device failure. For the core exercise, use exactly the predictor list in Task 2 and do not change it after seeing holdout results. The untouched holdout is for evaluating this prespecified workflow, not for repeatedly choosing better models.

## Task 1 — Inspect the prediction unit (5 min)
Load the pairs as `pairs`; verify 264 rows, 96 unique patients, and unique current `visit_id`. Inspect a single patient's current and target visit identifiers in chronological order. Confirm every target window is later than its current window and the planned horizon matches their difference.

Before execution, predict whether randomly splitting the 264 rows can put one patient's data on both sides. Explain why a target from one pair can be a current-visit feature in that patient's next pair, making a row split especially problematic.

## Task 2 — Freeze the feature and target definitions (5 min)
Create `X` using only these predictors:

- Numeric: `diag_mean_kohm`, `diag_sd_kohm`, `near_mean_kohm`, `far_mean_kohm`, `n_diag_valid`, `age_at_implant_years`, `months_since_implant`, `planned_horizon_months`.
- Categorical: `array_type`, `protocol`, `site`.

Create `y` from `target_diag_mean_kohm` and `groups` from `patient_id`. Keep IDs for auditing, outside `X`. Confirm 11 input columns and a nonmissing target. Inspect missingness but do not impute yet.

Explain why current protocol is allowed but target-visit information is not. Patient ID can serve as a memorization shortcut; it is not a transferable feature for this new-patient question.

## Task 3 — Hold out patients, not rows (7 min)
Use `GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)` to obtain development and holdout positional indices from `.split(X, y, groups)`. Slice X, y, and groups consistently with `.iloc`.

Acceptance: 72 development patients, 24 holdout patients, no patient overlap, and every pair belongs to exactly one partition. Row counts need not split exactly 75:25 because patients contribute different numbers of pairs.

Unrelated splitter pattern:
```python
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=7)
train_idx, test_idx = next(splitter.split(machine_X, machine_y, groups=machine_ids))
```
Record the holdout patient IDs and leave their outcomes unused until Task 7. Structural assertions on IDs and shapes are fine.

## Task 4 — Establish a persistence baseline (5 min)
On development rows only, predict the next visit's diagonal mean as the current `diag_mean_kohm`. Calculate MAE in kΩ, first overall and then by planned horizon. This rule has no fitted parameters.

Use `sklearn.metrics.mean_absolute_error`. Explain why a future-visit model must beat persistence to add value here, and why persistence can be strong when patients have stable personal baselines. Keep these development results separate from the later holdout result.

## Task 5 — Build a leakage-safe ridge pipeline (10 min)
Construct numeric preprocessing with median `SimpleImputer` followed by `StandardScaler`. Construct categorical preprocessing with most-frequent `SimpleImputer` followed by `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`. Combine them in a `ColumnTransformer`, then append `Ridge()` in a `Pipeline` with steps named `"preprocess"` and `"model"`.

Imports: `Pipeline` from `sklearn.pipeline`; `ColumnTransformer` from `sklearn.compose`; `SimpleImputer` from `sklearn.impute`; `StandardScaler`, `OneHotEncoder` from `sklearn.preprocessing`; `Ridge` from `sklearn.linear_model`.

Unrelated construction pattern:
```python
numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
preprocess_example = ColumnTransformer([
    ("numeric", numeric_steps, ["temperature", "pressure"]),
])
example = Pipeline([("preprocess", preprocess_example), ("model", Ridge())])
```
The task's categorical branch is separate. Do not fit preprocessing on all data first. Explain why scaling affects a penalty even when unpenalized linear regression can be reparameterized without changing fitted predictions.

## Task 6 — Tune with patient-grouped cross-validation (7 min)
Use `GroupKFold(n_splits=4)` inside `GridSearchCV` to compare `model__alpha` values `[0.1, 1.0, 10.0, 100.0]` with `scoring="neg_mean_absolute_error"` and `refit=True`. Fit using development X/y and pass **development patient IDs** through `groups=`.

Before fitting, iterate over `cv.split(X_dev, y_dev, groups_dev)` and assert no patient overlap in each train/validation fold. After fitting, report `best_params_` and convert `best_score_` to positive MAE. Explain that `.best_estimator_` is already refitted on all development rows.

Unrelated API reminder:
```python
search = GridSearchCV(pipe, {"model__alpha": [1, 10]}, cv=group_cv,
                      scoring="neg_mean_absolute_error")
search.fit(X_train, y_train, groups=train_machine_ids)
```
These CV scores support model selection. They are not the final holdout estimate.

## Task 7 — Evaluate once on held-out patients (7 min)
Now predict holdout rows with the selected pipeline and with persistence. Report both MAEs and their difference (model minus persistence). A negative difference favors the model. Do not require the model to win.

Build an evaluation table with patient ID, planned horizon, observed outcome, both predictions, and both absolute errors. Also calculate each patient's mean absolute error first and then average those patient means for each method. Explain the difference between weighting visit pairs equally and weighting patients equally.

Use the same observed rows for both methods. Do not retune after viewing these results; propose future experiments separately.

## Task 8 — Inspect errors and write a modeling decision (6 min)
Create an observed-versus-predicted scatterplot for the ridge model with an identity line and equal axis limits. Summarize holdout MAE by planned horizon and report the patient/row counts contributing to each.

Write a short decision: did ridge add value over persistence under the chosen metric; where is it weakest; and what change would you investigate using development data in a new experiment? Include why this synthetic result cannot establish performance in a real clinical dataset or at a new site.

## Retrieval practice
Audit this suggestion without running it:
```python
X_scaled = StandardScaler().fit_transform(X_numeric)
train_test_split(X_scaled, y, test_size=0.25, random_state=42)
```
Identify two separate leakage problems for this dataset and describe the appropriate replacement for each. Then name one extra evaluation change needed to forecast later calendar periods rather than generalize to unseen patients.

## Professional reasoning
1. Why is grouped splitting still necessary when `patient_id` is excluded from the predictors?
2. What could go wrong if the benchmark only includes people who return for follow-up, but the deployed population includes everyone?
3. Why does predicting next-visit telemetry not establish that telemetry predicts hearing performance or complications?

## Self-check and reflection
Record confidence **1–5**, the hardest point, one operation you can now explain, what needs repetition, and whether the workload was too light / about right / too heavy.

Finish line: restart the kernel, run your cells in order, and write a short research conclusion with its limitations. Ask for review of your attempt.

## Optional extension — outside the core hour
**25–40 minutes:** Estimate uncertainty in the holdout MAE difference by resampling **patients**, carrying all their pairs together. Prespecify whether the statistic weights patients or pairs equally and use a fixed seed; do not bootstrap individual rows as independent observations. This interval is conditional on the fitted workflow and does not include all model-selection uncertainty.

Alternatively, on development data only, compare your summaries with a prespecified set of off-diagonal matrix features or PCA inside the pipeline. Keep patient grouping, contact order, QC, and missing-value handling explicit. If the holdout has already influenced your design, label a reused-holdout comparison exploratory and plan a fresh external evaluation.

Reference: [scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data).